# 4. 서비스 개선 효과를 객관적으로 확인하기

2번 노트북에서 이미 기존 RAG 답변 데이터셋을 만들었고, 3번 노트북에서 그 답변을 RAGAS로 평가했습니다.

따라서 이 노트북에서는 기존 RAG를 다시 구축하지 않습니다. 대신 **새로운 chunk 전략만 추가로 실행**한 뒤, 기존 결과와 비교합니다.

핵심 흐름은 다음과 같습니다.

1. 2번/3번 노트북의 기존 결과를 기준선으로 로드합니다.
2. 새로운 chunk 전략으로 Vector DB를 새로 만듭니다.
3. 같은 질문, 같은 프롬프트, 같은 답변 모델, 같은 `k`로 답변을 생성합니다.
4. 새 답변을 RAGAS로 평가합니다.
5. 기존 평가 결과와 새 평가 결과를 비교합니다.


## 비교할 실험 조건

비교의 목적은 chunk 전략 변경 효과를 보는 것입니다. 그래서 답변 모델, 프롬프트, 검색 개수는 2번 노트북과 동일하게 유지합니다.

| 구분 | 산출 위치 | chunk_size | chunk_overlap | retriever k | 답변 모델 |
| --- | --- | ---: | ---: | ---: | --- |
| 기존 기준선 | 2번/3번 노트북 결과 CSV | 500 | 50 | 4 | `gemma3:4b` |
| 새 chunk 전략 | 이 노트북에서 새로 생성 | 300 | 80 | 4 | `gemma3:4b` |

새 전략의 점수가 더 높을 수도 있고 낮을 수도 있습니다. 중요한 것은 같은 평가셋과 같은 지표로 실제 실행 결과를 비교하는 것입니다.


In [1]:
from __future__ import annotations

import ast
import json
from pathlib import Path

import pandas as pd
from datasets import Dataset
from dotenv import load_dotenv
from IPython.display import display
from langchain_chroma import Chroma
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_ollama.chat_models import ChatOllama
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from ragas import evaluate
from ragas.metrics import answer_relevancy, context_precision, context_recall, faithfulness

load_dotenv()

pd.set_option("display.max_colwidth", 120)

DATA_DIR = Path("./data")
PDF_PATH = DATA_DIR / "SPRI_AI_Brief_2023년12월호_F.pdf"
BASELINE_ANSWERS_PATH = DATA_DIR / "evaluation_answers_from_rag.csv"
BASELINE_SCORES_PATH = DATA_DIR / "evaluation_scores_from_ragas.csv"

NEW_ANSWERS_PATH = DATA_DIR / "evaluation_answers_chunk_strategy.csv"
NEW_SCORES_PATH = DATA_DIR / "evaluation_scores_chunk_strategy.csv"
COMPARISON_PATH = DATA_DIR / "evaluation_chunk_strategy_comparison.csv"

METRICS = [
    "answer_relevancy",
    "faithfulness",
    "context_recall",
    "context_precision",
]


In [2]:
def parse_list_column(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []

    text = str(value)
    for parser in (json.loads, ast.literal_eval):
        try:
            parsed = parser(text)
            return parsed if isinstance(parsed, list) else [str(parsed)]
        except (json.JSONDecodeError, ValueError, SyntaxError):
            continue
    return [text]


baseline_answers = pd.read_csv(BASELINE_ANSWERS_PATH)
baseline_scores = pd.read_csv(BASELINE_SCORES_PATH)

required_answer_columns = {"user_input", "reference", "reference_contexts"}
required_score_columns = {"user_input", *METRICS, "min_score", "diagnosis", "response", "reference"}

missing_answer_columns = required_answer_columns - set(baseline_answers.columns)
missing_score_columns = required_score_columns - set(baseline_scores.columns)

if missing_answer_columns:
    raise ValueError(f"기존 답변 CSV에 필수 컬럼이 없습니다: {sorted(missing_answer_columns)}")
if missing_score_columns:
    raise ValueError(f"기존 평가 CSV에 필수 컬럼이 없습니다: {sorted(missing_score_columns)}")

baseline_answers["reference_contexts"] = baseline_answers["reference_contexts"].apply(parse_list_column)

eval_questions = baseline_answers[["user_input", "reference", "reference_contexts"]].copy()

print(f"기존 기준선 답변 수: {len(baseline_answers)}")
print(f"기존 기준선 평가 수: {len(baseline_scores)}")
display(eval_questions.head())


기존 기준선 답변 수: 15
기존 기준선 평가 수: 15


,user_input,reference,reference_contexts
0,미국에서 AI의 안전하고 신뢰할 수 있는 개발과 사용에 관한 행정명령은 어떤 내용이 포함되어 있습니까?,"미국 바이든 대통령이 서명한 '안전하고 신뢰할 수 있는 AI 개발과 사용에 관한 행정명령'은 AI의 안전과 보안 기준 마련, 개인정보보호 등 광범위한 행정 조치를 명시하고 있습니다.","[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n 미국, 안전하고 신뢰할 수 있는 AI 개발과 사용에 관한 행정명령 발표\n KEY Contents\nn 미국 바이든 대통령이 ‘안전하고 신..."
1,바이든 대통령의 AI 행정명령은 AI의 안전하고 신뢰할 수 있는 개발과 활용을 보장하기 위해 어떤 주요 내용을 포함하고 있습니까?,"바이든 대통령의 AI 행정명령은 AI의 안전과 보안 기준 마련, 개인정보보호, 형평성과 시민권 향상, 소비자 보호, 노동자 지원, 혁신과 경쟁 촉진, 국제협력에 관한 내용을 포괄합니다. 이 행정명령은 강력한 ...","[형평성과 시민권 향상 △소비자\n보호 △노동자 지원 △혁신과 경쟁 촉진 △국제협력을 골자로 함\n£바이든 대통령, AI 행정명령 통해 안전하고 신뢰할 수 있는 AI 개발과 활용 추진\nn 미국 바이든 대통령..."
2,NAIRR 뭐하는거야?,"NAIRR은 국가AI연구자원(National Artificial Intelligence Research Resource)으로, 미국 전역의 AI 연구를 촉진하고 중소기업과 개발자에게 기술과 인프라를 지원하는 ...","[혁신과 경쟁 촉진 △국제협력을 골자로 함 £바이든 대통령, AI 행정명령 통해 안전하고 신뢰할 수 있는 AI 개발과 활용 추진 n 미국 바이든 대통령이 2023년 10월 30일 연방정부 차원에서 안전하고 신..."
3,The White House의 2023년 10월 30일자 행정명령은 미국에서 외국인들이 AI 분야에서 공부하고 취업할 수 있도록 어떤 지원을 제공하고 있습니까?,The White House의 행정명령은 분야의 전문 지식을 갖춘 외국인들이 미국에서 공부하고 취업할 수 있도록 지원하는 내용을 포함하고 있습니다.,"[분야의 전문 지식을 갖춘 외국인들이 미국에서 공부하고 취업할 수 있도록 지원 ☞ 출처 : The White House, Executive Order on the Safe, Secure, and Trustwo..."
4,G7의 AI 국제 행동강령은 AI 시스템의 안전과 보안 기능을 어떻게 보장하나요?,"G7의 AI 국제 행동강령은 AI 시스템의 안전과 보안 기능을 보장하기 위해 AI 수명주기 전반에 걸친 위험 평가와 완화, 투명성과 책임성의 보장, 정보공유와 이해관계자 간 협력, 보안 통제, 콘텐츠 인증과 ...",[<1-hop>\n\n정보 교류 활성화를 핵심 기능으로 함\nn (첨단 AI 시스템 평가 개발과 시행) 시스템의 안전 관련 속성을 중심으로 안전과 보안 기능을 이해\n하고 사회적 영향을 평가\n∙ 평가 우선순...


## 새 chunk 전략으로 Vector DB 만들기

기존 결과는 그대로 두고, 새 chunk 전략용 Chroma collection만 새로 만듭니다.

기존 기준선과 비교하기 위해 `retriever k`는 2번 노트북과 같은 4로 유지합니다.


In [3]:
NEW_CHUNK_SIZE = 300
NEW_CHUNK_OVERLAP = 80
RETRIEVER_K = 4

CHROMA_PATH = "./chroma_chunk_strategy"
COLLECTION_NAME = "chunk_size_300_overlap_80"

documents = PDFPlumberLoader(str(PDF_PATH)).load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=NEW_CHUNK_SIZE,
    chunk_overlap=NEW_CHUNK_OVERLAP,
)
split_documents = text_splitter.split_documents(documents)

print(f"PDF 페이지 수: {len(documents)}")
print(f"새 chunk 전략의 분할 문서 수: {len(split_documents)}")


PDF 페이지 수: 23
새 chunk 전략의 분할 문서 수: 129


In [4]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 같은 collection을 재실행할 때 문서가 중복 저장되지 않도록 collection을 초기화합니다.
existing_vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=CHROMA_PATH,
)
existing_vectorstore.delete_collection()

vectorstore = Chroma.from_documents(
    documents=split_documents,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_PATH,
)

retriever = vectorstore.as_retriever(search_kwargs={"k": RETRIEVER_K})
print(f"검색 문맥 수(k): {RETRIEVER_K}")


검색 문맥 수(k): 4


## 새 chunk 전략으로 답변 생성

프롬프트와 답변 모델은 2번 노트북과 같은 구조를 사용합니다. 이렇게 해야 chunk 전략 변경 효과를 더 깨끗하게 비교할 수 있습니다.


In [5]:
prompt = PromptTemplate.from_template(
    template="""
    - 당신은 질의응답 작업을 위한 어시스턴트입니다.
    - 아래에 제공된 검색/추출된 컨텍스트를 활용하여 질문에 답하십시오.
    - 답을 모를 경우, 모른다고 말하십시오.
    - 반드시 한국어로 답변해야 합니다.

    #컨텍스트:
    {context}

    #질문:
    {question}
    """
)

llm = ChatOllama(
    model="gemma3:4b",
    temperature=0.1,
    top_p=1.0,
    num_predict=256,
    keep_alive="5m",
)

answer_chain = prompt | llm | StrOutputParser()


In [6]:
def generate_answer(question: str) -> dict:
    retrieved_docs = retriever.invoke(question)
    contexts = [doc.page_content for doc in retrieved_docs]
    response = answer_chain.invoke(
        {
            "context": "\n".join(contexts),
            "question": question,
        }
    )
    return {
        "retrieved_contexts": contexts,
        "response": response,
    }


new_answer_rows = []

for row in eval_questions.itertuples(index=False):
    result = generate_answer(row.user_input)
    new_answer_rows.append(
        {
            "user_input": row.user_input,
            "retrieved_contexts": result["retrieved_contexts"],
            "response": result["response"],
            "reference": row.reference,
            "reference_contexts": row.reference_contexts,
        }
    )

new_answers = pd.DataFrame(new_answer_rows)
display(new_answers[["user_input", "response"]].head())


,user_input,response
0,미국에서 AI의 안전하고 신뢰할 수 있는 개발과 사용에 관한 행정명령은 어떤 내용이 포함되어 있습니까?,미국에서 AI의 안전하고 신뢰할 수 있는 개발과 사용에 관한 행정명령은 다음과 같은 내용을 포함하고 있습니다:\n\n* AI의 안전 및 보안 기준 마련\n* 개인정보 보호\n* 형평성과 시민권 향상...
1,바이든 대통령의 AI 행정명령은 AI의 안전하고 신뢰할 수 있는 개발과 활용을 보장하기 위해 어떤 주요 내용을 포함하고 있습니까?,바이든 대통령의 AI 행정명령은 다음과 같은 주요 내용을 포함하고 있습니다.\n\n* AI의 안전 및 보안 기준 마련\n* 개인정보 보호\n* 형평성 및 시민권 향상\n* 소비자 보호\n* ...
2,NAIRR 뭐하는거야?,"NAIRR(국가AI연구자원)은 미국 전역의 AI 연구를 촉진하고, 중소기업과 개발자에 기술과 인프라를 지원하는 국가 차원의 AI 연구 인프라 확충 프로그램입니다."
3,The White House의 2023년 10월 30일자 행정명령은 미국에서 외국인들이 AI 분야에서 공부하고 취업할 수 있도록 어떤 지원을 제공하고 있습니까?,The White House의 2023년 10월 30일자 행정명령은 비자 기준과 인터뷰 절차를 현대화하고 간소화하여 AI 관련 주요 분야의 전문 지식을 갖춘 외국인들이 미국에서 공부하고 취업할 수 있도록 지원...
4,G7의 AI 국제 행동강령은 AI 시스템의 안전과 보안 기능을 어떻게 보장하나요?,"G7의 AI 국제 행동강령은 AI 수명주기 전반에 걸친 위험 평가와 완화, 투명성과 책임성의 보장을 통해 AI 시스템의 안전과 보안 기능을 보장합니다. 또한, 정보공유와 이해관계자 간 협력, 보안 통제, 콘텐..."


In [7]:
new_answers_for_csv = new_answers.copy()
for column in ["retrieved_contexts", "reference_contexts"]:
    new_answers_for_csv[column] = new_answers_for_csv[column].apply(
        lambda value: json.dumps(value, ensure_ascii=False)
    )

new_answers_for_csv.to_csv(NEW_ANSWERS_PATH, index=False, encoding="utf-8-sig")
print(f"저장 완료: {NEW_ANSWERS_PATH}")


저장 완료: data\evaluation_answers_chunk_strategy.csv


## 새 답변 RAGAS 평가

새 chunk 전략으로 만든 답변만 RAGAS로 평가합니다. 기존 평가는 3번 노트북에서 만든 결과를 재사용합니다.


In [8]:
dataset_new = Dataset.from_pandas(
    new_answers[["user_input", "retrieved_contexts", "response", "reference", "reference_contexts"]],
    preserve_index=False,
)

judge_llm = ChatOpenAI(model="gpt-4o-mini")

result_new = evaluate(
    dataset=dataset_new,
    metrics=[answer_relevancy, faithfulness, context_recall, context_precision],
    llm=judge_llm,
)

new_scores = result_new.to_pandas()
new_scores[METRICS] = new_scores[METRICS].apply(pd.to_numeric, errors="coerce")
new_scores["min_score"] = new_scores[METRICS].min(axis=1)
display(new_scores[["user_input", *METRICS, "min_score"]].head())


Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


,user_input,answer_relevancy,faithfulness,context_recall,context_precision,min_score
0,미국에서 AI의 안전하고 신뢰할 수 있는 개발과 사용에 관한 행정명령은 어떤 내용이 포함되어 있습니까?,0.884432,1.0,1.0,1.0,0.884432
1,바이든 대통령의 AI 행정명령은 AI의 안전하고 신뢰할 수 있는 개발과 활용을 보장하기 위해 어떤 주요 내용을 포함하고 있습니까?,0.850969,1.0,1.0,1.0,0.850969
2,NAIRR 뭐하는거야?,0.845553,1.0,1.0,1.0,0.845553
3,The White House의 2023년 10월 30일자 행정명령은 미국에서 외국인들이 AI 분야에서 공부하고 취업할 수 있도록 어떤 지원을 제공하고 있습니까?,0.834006,0.5,1.0,1.0,0.500000
4,G7의 AI 국제 행동강령은 AI 시스템의 안전과 보안 기능을 어떻게 보장하나요?,0.890596,0.8,0.5,1.0,0.500000


In [9]:
LOW_SCORE_THRESHOLD = 0.70


def diagnose_rag_failure(row: pd.Series) -> str:
    retrieval_low = min(row["context_recall"], row["context_precision"]) < LOW_SCORE_THRESHOLD
    generation_low = min(row["answer_relevancy"], row["faithfulness"]) < LOW_SCORE_THRESHOLD

    if retrieval_low and generation_low:
        return "검색과 생성 모두 점검"
    if retrieval_low:
        return "검색 단계 우선 점검"
    if generation_low:
        return "생성 단계 우선 점검"
    return "큰 문제 없음"


new_scores["diagnosis"] = new_scores.apply(diagnose_rag_failure, axis=1)

score_output_columns = [
    "user_input",
    *METRICS,
    "min_score",
    "diagnosis",
    "response",
    "reference",
]

new_scores[score_output_columns].to_csv(NEW_SCORES_PATH, index=False, encoding="utf-8-sig")
print(f"저장 완료: {NEW_SCORES_PATH}")


저장 완료: data\evaluation_scores_chunk_strategy.csv


## 기존 결과와 새 chunk 전략 비교

이제 3번 노트북에서 만든 기존 평가 결과와, 이 노트북에서 만든 새 chunk 전략 평가 결과를 비교합니다.


In [10]:
baseline_labeled = baseline_scores.copy()
baseline_labeled["config_name"] = "기존 기준선: 2번 노트북 결과"

new_labeled = new_scores[score_output_columns].copy()
new_labeled["config_name"] = "새 chunk 전략: size=300, overlap=80"

comparison_scores = pd.concat([baseline_labeled, new_labeled], ignore_index=True)

summary = (
    comparison_scores
    .assign(is_low_score=lambda df: df["min_score"] < LOW_SCORE_THRESHOLD)
    .groupby("config_name")
    .agg(
        answer_relevancy=("answer_relevancy", "mean"),
        faithfulness=("faithfulness", "mean"),
        context_recall=("context_recall", "mean"),
        context_precision=("context_precision", "mean"),
        min_score=("min_score", "mean"),
        low_score_count=("is_low_score", "sum"),
        sample_count=("user_input", "count"),
    )
    .round(4)
)

display(summary)


,answer_relevancy,faithfulness,context_recall,context_precision,min_score,low_score_count,sample_count
config_name,,,,,,,
기존 기준선: 2번 노트북 결과,0.7495,0.8491,0.9389,0.8167,0.6373,7,15
"새 chunk 전략: size=300, overlap=80",0.6869,0.8654,0.7778,0.8667,0.5680,8,15


In [15]:
baseline_name = "기존 기준선: 2번 노트북 결과"
new_name = "새 chunk 전략: size=300, overlap=80"

delta = (
    summary.loc[new_name, METRICS + ["min_score"]]
    - summary.loc[baseline_name, METRICS + ["min_score"]]
).to_frame("변화량")

metric_descriptions = {
    "answer_relevancy": "답변이 질문 의도에 얼마나 직접적으로 맞는지",
    "faithfulness": "답변이 검색된 문맥에 근거해 사실적으로 일관되는지",
    "context_recall": "기준 정답에 필요한 근거가 검색 문맥에 충분히 포함됐는지",
    "context_precision": "검색된 문맥 중 질문과 기준 정답에 관련 있는 내용이 얼마나 상위에 있는지",
    "min_score": "각 샘플의 네 지표 중 가장 낮은 점수의 평균으로, 전체 품질의 약한 고리를 보는 값",
}

delta.insert(0, "지표 설명", pd.Series(metric_descriptions))
delta["판단"] = delta["변화량"].apply(lambda value: "개선" if value > 0 else "하락/동일")
display(delta.round(4))


,지표 설명,변화량,판단
answer_relevancy,답변이 질문 의도에 얼마나 직접적으로 맞는지,-0.0626,하락/동일
faithfulness,답변이 검색된 문맥에 근거해 사실적으로 일관되는지,0.0163,개선
context_recall,기준 정답에 필요한 근거가 검색 문맥에 충분히 포함됐는지,-0.1611,하락/동일
context_precision,검색된 문맥 중 질문과 기준 정답에 관련 있는 내용이 얼마나 상위에 있는지,0.0500,개선
min_score,"각 샘플의 네 지표 중 가장 낮은 점수의 평균으로, 전체 품질의 약한 고리를 보는 값",-0.0693,하락/동일


## 질문별 변화 확인

평균 점수가 좋아져도 특정 질문에서는 나빠질 수 있습니다. 질문별 변화량을 확인해 어떤 유형에서 새 chunk 전략이 유리하거나 불리한지 봅니다.


In [12]:
question_delta = baseline_scores[["user_input", *METRICS, "min_score"]].merge(
    new_scores[["user_input", *METRICS, "min_score"]],
    on="user_input",
    how="inner",
    suffixes=("_baseline", "_new"),
)

for metric in [*METRICS, "min_score"]:
    question_delta[f"{metric}_delta"] = (
        question_delta[f"{metric}_new"] - question_delta[f"{metric}_baseline"]
    )

delta_columns = [
    "user_input",
    *[f"{metric}_delta" for metric in [*METRICS, "min_score"]],
]

display(
    question_delta[delta_columns]
    .sort_values("min_score_delta")
    .round(4)
)


,user_input,answer_relevancy_delta,faithfulness_delta,context_recall_delta,context_precision_delta,min_score_delta
11,"EU AI 법 협상에서 프랑스, 독일, 이탈리아의 제안이 '기반모델 전반 규제 반대'와 '자율 행동강령 의무화' 조건을 모두 만족한다면, 감독기관은 준수 여부와 제재를 어떻게 다루어야 하나요?",-0.8500,-0.1667,-1.0000,0.00,-0.6667
8,영국의 AI 안전 연구소가 글로벌 허브로 자리잡기 위해 어떤 국제 협력 방안을 추진하고 있나요?,-0.0173,-0.3333,-0.4167,0.00,-0.4167
4,G7의 AI 국제 행동강령은 AI 시스템의 안전과 보안 기능을 어떻게 보장하나요?,0.0086,-0.2000,-0.5000,0.00,-0.3820
9,영국 AI 안전 연구소는 첨단 AI 시스템 평가와 관련하여 어떤 정책/법제를 추진하고 있나요?,-0.0430,0.1667,-0.5000,0.00,-0.3333
3,The White House의 2023년 10월 30일자 행정명령은 미국에서 외국인들이 AI 분야에서 공부하고 취업할 수 있도록 어떤 지원을 제공하고 있습니까?,-0.0802,-0.1667,0.0000,0.00,-0.1667
1,바이든 대통령의 AI 행정명령은 AI의 안전하고 신뢰할 수 있는 개발과 활용을 보장하기 위해 어떤 주요 내용을 포함하고 있습니까?,-0.0013,0.0000,0.0000,0.00,-0.0013
0,미국에서 AI의 안전하고 신뢰할 수 있는 개발과 사용에 관한 행정명령은 어떤 내용이 포함되어 있습니까?,-0.0010,0.0833,0.0000,0.00,-0.0010
6,프런티어 모델 포럼이 조성한 AI 안전 기금의 목적은 무엇인가요?,0.0000,0.0000,0.0000,0.00,0.0000
14,"LG Aimers 4기의 교육 일정, 지원 자격, 과제 제출 방식을 알려 주세요.",0.0000,0.0000,0.0000,0.00,0.0000
13,이 문서만 근거로 2024년에 최종 통과된 EU AI Act의 조항과 시행 일정을 설명해 주세요.,0.0000,0.2000,0.0000,0.00,0.0000


In [13]:
comparison_scores[
    [
        "config_name",
        "user_input",
        *METRICS,
        "min_score",
        "diagnosis",
        "response",
        "reference",
    ]
].to_csv(COMPARISON_PATH, index=False, encoding="utf-8-sig")

print(f"저장 완료: {COMPARISON_PATH}")


저장 완료: data\evaluation_chunk_strategy_comparison.csv


## 낮은 점수 샘플 확인

평균 점수만으로는 개선 여부를 판단하지 않습니다. 새 chunk 전략에서 여전히 낮은 점수로 남은 질문을 직접 확인합니다.


In [14]:
low_cases_new = (
    new_labeled
    .loc[new_labeled["min_score"] < LOW_SCORE_THRESHOLD, ["user_input", *METRICS, "min_score", "diagnosis", "response", "reference"]]
    .sort_values("min_score")
)

print(f"새 chunk 전략의 낮은 점수 샘플 수: {len(low_cases_new)}")
display(low_cases_new)


새 chunk 전략의 낮은 점수 샘플 수: 8


,user_input,answer_relevancy,faithfulness,context_recall,context_precision,min_score,diagnosis,response,reference
13,이 문서만 근거로 2024년에 최종 통과된 EU AI Act의 조항과 시행 일정을 설명해 주세요.,0.000000,0.800000,1.000000,0.0,0.000000,검색과 생성 모두 점검,"제공된 문서에는 2024년에 최종 통과된 EU AI Act의 조항이나 시행 일정에 대한 정보가 없습니다. \n\n문서에서는 유럽의회, EU 집행위원회, EU 이사회가 ‘AI 법(AI act)’에 대한 최종 협...","제공된 SPRI 2023년 12월 문서는 EU AI 법의 최종 통과 이후 조항이나 시행 일정을 다루지 않습니다. 문서에는 2023년 10월 24일 3자 협상 회의의 기본 합의, 2023년 11월 10일 통신작..."
14,"LG Aimers 4기의 교육 일정, 지원 자격, 과제 제출 방식을 알려 주세요.",0.000000,1.000000,0.666667,0.0,0.000000,검색과 생성 모두 점검,"제공된 컨텍스트에는 LG Aimers 4기 관련 정보가 없습니다. 컨텍스트는 AI 발전 협회(AAAI) 컨퍼런스 및 구글 딥마인드, 갈릴레오 관련 내용과 인력/교육 관련 정보를 담고 있습니다. 따라서 LG A...","현재 평가용 RAG 코퍼스는 SPRI AI Brief 2023년 12월호 문서를 사용합니다. 이 문맥에는 LG Aimers 4기의 교육 일정, 지원 자격, 과제 제출 방식이 포함되어 있지 않습니다. 문서 기반..."
11,"EU AI 법 협상에서 프랑스, 독일, 이탈리아의 제안이 '기반모델 전반 규제 반대'와 '자율 행동강령 의무화' 조건을 모두 만족한다면, 감독기관은 준수 여부와 제재를 어떻게 다루어야 하나요?",0.000000,0.500000,0.000000,1.0,0.000000,검색과 생성 모두 점검,"제공된 컨텍스트에 따르면, 프랑스, 독일, 이탈리아는 기반모델에 대한 모든 유형의 규제에 반대하며, 자율적 행동강령 도입 및 준수를 의무화하는 방안을 제안했습니다. 이러한 제안이 최종적으로 합의에 이르렀다면,...","프랑스, 독일, 이탈리아는 기반모델 전반 규제가 기술 중립적이고 위험 기반의 AI 규제 원칙에 어긋난다고 보고, 특정 용도로 사용될 수 있는 AI 시스템에 대한 규제를 요구했습니다. 동시에 자발적인 행동강령을..."
8,영국의 AI 안전 연구소가 글로벌 허브로 자리잡기 위해 어떤 국제 협력 방안을 추진하고 있나요?,0.864618,0.666667,0.333333,1.0,0.333333,검색과 생성 모두 점검,"영국 AI 안전 연구소는 개발, 공공부문 역량 구축과 과학 연구개발 등의 분야에서 협력하기로 합의했습니다. 또한, 리시 수낙 영국 총리는 AI 안전성 정상회의를 통해 첨단 AI 모델에 대한 안전성 시험 계획 ...","영국의 AI 안전 연구소는 AI 안전성 정상회의에서 발표된 블레츨리 선언을 통해 국제 협력 방안을 추진하고 있습니다. 이 선언은 AI 안전 보장을 위해 국가, 국제기구, 기업, 시민사회, 학계 등 모든 이해관..."
9,영국 AI 안전 연구소는 첨단 AI 시스템 평가와 관련하여 어떤 정책/법제를 추진하고 있나요?,0.854465,1.000000,0.500000,1.0,0.500000,검색 단계 우선 점검,"영국 AI 안전 연구소는 첨단 AI 시스템 평가 개발과 시행을 통해 안전 관련 속성을 중심으로 안전 및 보안 기능을 이해하고 사회적 영향을 평가하는 것을 핵심 기능으로 추진하고 있습니다. 또한, 사이버범죄 조...","영국 AI 안전 연구소는 첨단 AI 시스템 평가 개발과 시행을 통해 안전성을 보장하고, AI 안전 연구를 촉진하며, 정보 교류를 활성화하는 정책/법제를 추진하고 있습니다. 연구소는 외부 기관과 협력하여 시스템..."
4,G7의 AI 국제 행동강령은 AI 시스템의 안전과 보안 기능을 어떻게 보장하나요?,0.890596,0.800000,0.500000,1.0,0.500000,검색 단계 우선 점검,"G7의 AI 국제 행동강령은 AI 수명주기 전반에 걸친 위험 평가와 완화, 투명성과 책임성의 보장을 통해 AI 시스템의 안전과 보안 기능을 보장합니다. 또한, 정보공유와 이해관계자 간 협력, 보안 통제, 콘텐...","G7의 AI 국제 행동강령은 AI 시스템의 안전과 보안 기능을 보장하기 위해 AI 수명주기 전반에 걸친 위험 평가와 완화, 투명성과 책임성의 보장, 정보공유와 이해관계자 간 협력, 보안 통제, 콘텐츠 인증과 ..."
3,The White House의 2023년 10월 30일자 행정명령은 미국에서 외국인들이 AI 분야에서 공부하고 취업할 수 있도록 어떤 지원을 제공하고 있습니까?,0.834006,0.500000,1.000000,1.0,0.500000,생성 단계 우선 점검,The White House의 2023년 10월 30일자 행정명령은 비자 기준과 인터뷰 절차를 현대화하고 간소화하여 AI 관련 주요 분야의 전문 지식을 갖춘 외국인들이 미국에서 공부하고 취업할 수 있도록 지원...,The White House의 행정명령은 분야의 전문 지식을 갖춘 외국인들이 미국에서 공부하고 취업할 수 있도록 지원하는 내용을 포함하고 있습니다.
10,"AI 행정명령과 G7 국제 행동강령을 비교해, 둘 다 공통으로 요구하는 위험 관리 조치 2가지와 G7 행동강령에 더 구체적으로 나온 조치 1가지를 구분해서 답하세요.",0.840201,0.714286,0.666667,1.0,0.666667,검색 단계 우선 점검,## AI 행정명령과 G7 국제 행동강령 비교\n\n**공통으로 요구하는 위험 관리 조치 2가지:**\n\n1. **AI 수명주기 전반에 걸친 위험 평가 및 완화:** 두 모두 첨단 AI 시스템 개발 과정에...,두 문서 모두 첨단 AI 시스템의 위험을 평가하고 완화하는 조치와 투명성/책임성 강화를 요구합니다. 미국 AI 행정명령은 강력한 AI 시스템 개발 기업이 안전 테스트 결과와 주요 정보를 미국 정부와 공유하도록...
